In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
consumer_df = pd.read_csv("../data/raw/consumer_dataset_raw.csv")
population_df = pd.read_csv("../data/raw/us_ev_population_raw.csv")
iea_df = pd.read_csv("../data/raw/iea_global_ev_raw.csv")
stock_df = pd.read_csv("../data/raw/stock_details_raw.csv")
sales_df = pd.read_excel("../data/raw/global_ev_growth.xlsx")
charging_df = pd.read_csv("../data/raw/openchargemap_global_raw.csv")
survey_df = pd.read_csv("../data/raw/survey_details.csv")

/var/folders/fp/w6yn5n6d4cb299k1w5hy5xgc0000gn/T/ipykernel_91314/3521166484.py:6: DtypeWarning: Columns (36,37) have mixed types. Specify dtype option on import or set low_memory=False.
  charging_df = pd.read_csv("../data/raw/openchargemap_global_raw.csv")


### IEA EV Dataset Cleaning:

In [15]:
iea_df.columns

Index(['region', 'category', 'parameter', 'mode', 'powertrain', 'year', 'unit',
       'value'],
      dtype='object')

In [4]:
iea_df.columns = iea_df.columns.str.lower()

iea_clean = iea_df[
    iea_df["parameter"].isin([
        "EV sales",
        "EV stock",
        "EV charging points"
    ])
]

In [5]:
iea_clean = iea_clean[
    iea_clean["mode"] == "Cars"
]

iea_clean = iea_clean[[
    "region",
    "parameter",
    "powertrain",
    "year",
    "value"
]]

In [6]:
iea_clean.isna().sum()

region        0
parameter     0
powertrain    0
year          0
value         0
dtype: int64

In [20]:
print("Original rows:", iea_df.shape[0])
print("Filtered rows:", iea_clean.shape[0])

Original rows: 12654
Filtered rows: 2975


In [7]:
iea_clean.to_csv("../data/processed/iea_ev_clean.csv", index=False)

### EV Population Dataset Cleaning:

In [9]:
population_df.columns

Index(['vin (1-10)', 'county', 'city', 'state', 'postal code', 'model year',
       'make', 'model', 'electric vehicle type',
       'clean alternative fuel vehicle (cafv) eligibility', 'electric range',
       'legislative district', 'dol vehicle id', 'vehicle location',
       'electric utility', '2020 census tract'],
      dtype='object')

In [11]:
population_df.columns = population_df.columns.str.lower()

ev_population_clean = population_df.groupby(
    ["model year", "make", "model", "electric vehicle type"]
).size().reset_index(name="vehicle_count")

In [12]:
ev_population_clean.isna().sum()

model year               0
make                     0
model                    0
electric vehicle type    0
vehicle_count            0
dtype: int64

In [13]:
ev_population_clean.to_csv("../data/processed/ev_population_clean.csv", index=False)

In [27]:
print("Original rows:", population_df.shape[0])
print("Filtered rows:", ev_population_clean.shape[0])

Original rows: 276828
Filtered rows: 714


### EV Sales Dataset Cleaning:

In [23]:
sales_df.columns

Index(['region_country', 'category', 'parameter', 'mode', 'powertrain', 'year',
       'unit', 'value', 'aggregate group'],
      dtype='object')

In [25]:
sales_df.columns = sales_df.columns.str.lower()

sales_clean = sales_df[[
    "region_country",
    "year",
    "value"
]].rename(columns={"value": "ev_sales"})

In [26]:
sales_clean.isna().sum()   

region_country    0
year              0
ev_sales          0
dtype: int64

In [28]:
print("Original rows:", sales_df.shape[0])
print("Filtered rows:", sales_clean.shape[0])

Original rows: 16436
Filtered rows: 16436


In [29]:
sales_clean.to_csv("../data/processed/ev_sales_clean.csv", index=False)

### Charging Infrastructure Dataset Cleaning:

In [30]:
charging_df.columns

Index(['IsRecentlyVerified', 'DateLastVerified', 'ID', 'UUID',
       'DataProviderID', 'OperatorID', 'UsageTypeID', 'Connections',
       'NumberOfPoints', 'StatusTypeID', 'DateLastStatusUpdate',
       'DataQualityLevel', 'DateCreated', 'SubmissionStatusTypeID',
       'AddressInfo.ID', 'AddressInfo.Title', 'AddressInfo.AddressLine1',
       'AddressInfo.Town', 'AddressInfo.StateOrProvince',
       'AddressInfo.Postcode', 'AddressInfo.CountryID', 'AddressInfo.Latitude',
       'AddressInfo.Longitude', 'AddressInfo.DistanceUnit',
       'AddressInfo.AddressLine2', 'AddressInfo.RelatedURL', 'UsageCost',
       'GeneralComments', 'AddressInfo.ContactTelephone1',
       'AddressInfo.ContactTelephone2', 'AddressInfo.AccessComments',
       'AddressInfo.ContactEmail', 'DataProvidersReference', 'DatePlanned',
       'country_code', 'OperatorsReference', 'DateLastConfirmed',
       'MetadataValues'],
      dtype='object')

In [31]:
charging_clean = charging_df[[
    "AddressInfo.CountryID",
    "AddressInfo.Latitude",
    "AddressInfo.Longitude",
    "NumberOfPoints"
]].copy()

charging_clean.columns = [
    "country",
    "latitude",
    "longitude",
    "charging_points"
]

In [32]:
charging_clean.isna().sum()

country                0
latitude               0
longitude              0
charging_points    20000
dtype: int64

In [33]:
print("Original rows:", charging_df.shape[0])
print("Filtered rows:", charging_clean.shape[0])

Original rows: 38219
Filtered rows: 38219


In [35]:
charging_clean["charging_points"] = charging_clean["charging_points"].fillna(1)

In [36]:
charging_clean["charging_points"] = charging_clean["charging_points"].astype(int)

In [37]:
charging_clean.isna().sum()

country            0
latitude           0
longitude          0
charging_points    0
dtype: int64

In [38]:
charging_clean.to_csv(
    "../data/processed/charging_infrastructure_clean.csv",
    index=False
)

### Stock Market Dataset Cleaning:

In [39]:
stock_df.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends',
       'Stock Splits', 'Company'],
      dtype='object')

In [40]:
stock_df.isna().sum()

Date            0
Open            0
High            0
Low             0
Close           0
Volume          0
Dividends       0
Stock Splits    0
Company         0
dtype: int64

In [41]:
ev_companies = ["TSLA", "NVDA", "NIO", "XPEV", "LI"]

stock_clean = stock_df[stock_df["Company"].isin(ev_companies)]

In [42]:
stock_clean["Date"] = pd.to_datetime(stock_clean["Date"])

/var/folders/fp/w6yn5n6d4cb299k1w5hy5xgc0000gn/T/ipykernel_91314/3248918533.py:1: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  stock_clean["Date"] = pd.to_datetime(stock_clean["Date"])
/var/folders/fp/w6yn5n6d4cb299k1w5hy5xgc0000gn/T/ipykernel_91314/3248918533.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  stock_clean["Date"] = pd.to_datetime(stock_clean["Date"])


In [43]:
stock_clean.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends',
       'Stock Splits', 'Company'],
      dtype='object')

In [44]:
stock_clean.to_csv("../data/processed/stock_market_clean.csv", index=False)

### Survey Dataset Cleaning:

In [51]:
survey_df.columns

Index(['country of residence', 'area type', 'age group',
       'annual income range (in local currency equivalent)\nplease refer your country's avg. national income',
       'occupation', 'do you currently own a vehicle?',
       'what type of vehicle do you currently own?',
       'if you own an ev, mention your ev brand',
       'if you do not own an ev, are you considering buying one in the next 2–3 years?',
       'average daily driving distance', 'lower running cost',
       'environmental benefits', 'government incentives / subsidies',
       'rising fuel prices', 'advanced technology & features',
       'brand reputation', 'charging convenience', 'evs are too expensive',
       'charging infrastructure is insufficient',
       'battery replacement cost is high', 'driving range is inadequate',
       'charging time is too long', 'ev resale value is uncertain',
       'limited service centers', 'preferred ev price range',
       'minimum acceptable driving range per charge',
    

In [52]:
survey_df.columns = (
    survey_df.columns
    .str.strip()
    .str.lower()
    .str.replace("\n", " ", regex=False)
    .str.replace(" ", "_")
    .str.replace("/", "_")
    .str.replace("?", "")
)

In [54]:
survey_df.isna().sum()

country_of_residence                                                                                    0
area_type                                                                                               0
age_group                                                                                               0
annual_income_range_(in_local_currency_equivalent)_please_refer_your_country's_avg._national_income     0
occupation                                                                                              0
do_you_currently_own_a_vehicle                                                                          0
what_type_of_vehicle_do_you_currently_own                                                               0
if_you_own_an_ev,_mention_your_ev_brand                                                                63
if_you_do_not_own_an_ev,_are_you_considering_buying_one_in_the_next_2–3_years                           0
average_daily_driving_distance                

In [55]:
survey_df["if_you_own_an_ev,_mention_your_ev_brand"] = \
survey_df["if_you_own_an_ev,_mention_your_ev_brand"].fillna("none")

In [56]:
motivation_cols = [
"lower_running_cost",
"environmental_benefits",
"government_incentives___subsidies",
"rising_fuel_prices",
"advanced_technology_&_features",
"brand_reputation",
"charging_convenience"
]

for col in motivation_cols:
    survey_df[col] = survey_df[col].fillna("moderately important")

In [57]:
barrier_cols = [
"evs_are_too_expensive",
"charging_infrastructure_is_insufficient",
"battery_replacement_cost_is_high",
"driving_range_is_inadequate",
"charging_time_is_too_long",
"ev_resale_value_is_uncertain",
"limited_service_centers"
]

for col in barrier_cols:
    survey_df[col] = survey_df[col].fillna("not important")

In [58]:
survey_df["primary_reason_for_resale___consideration"] = \
survey_df["primary_reason_for_resale___consideration"].fillna("not_applicable")

In [59]:
survey_df.isna().sum()

country_of_residence                                                                                    0
area_type                                                                                               0
age_group                                                                                               0
annual_income_range_(in_local_currency_equivalent)_please_refer_your_country's_avg._national_income     0
occupation                                                                                              0
do_you_currently_own_a_vehicle                                                                          0
what_type_of_vehicle_do_you_currently_own                                                               0
if_you_own_an_ev,_mention_your_ev_brand                                                                 0
if_you_do_not_own_an_ev,_are_you_considering_buying_one_in_the_next_2–3_years                           0
average_daily_driving_distance                

In [61]:
for col in survey_df.columns:
    print(col)

country_of_residence
area_type
age_group
annual_income_range_(in_local_currency_equivalent)_please_refer_your_country's_avg._national_income
occupation
do_you_currently_own_a_vehicle
what_type_of_vehicle_do_you_currently_own
if_you_own_an_ev,_mention_your_ev_brand
if_you_do_not_own_an_ev,_are_you_considering_buying_one_in_the_next_2–3_years
average_daily_driving_distance
lower_running_cost
environmental_benefits
government_incentives___subsidies
rising_fuel_prices
advanced_technology_&_features
brand_reputation
charging_convenience
evs_are_too_expensive
charging_infrastructure_is_insufficient
battery_replacement_cost_is_high
driving_range_is_inadequate
charging_time_is_too_long
ev_resale_value_is_uncertain
limited_service_centers
preferred_ev_price_range
minimum_acceptable_driving_range_per_charge
important_factors_when_choosing_an_ev
have_you_ever_sold_or_considered_selling_an_ev
primary_reason_for_resale___consideration
do_you_believe_evs_depreciate_faster_than_petrol_or_diesel_vehic

In [62]:
survey_df.head()

,country_of_residence,area_type,age_group,annual_income_range_(in_local_currency_equivalent)_please_refer_your_country's_avg._national_income,occupation,do_you_currently_own_a_vehicle,what_type_of_vehicle_do_you_currently_own,"if_you_own_an_ev,_mention_your_ev_brand","if_you_do_not_own_an_ev,_are_you_considering_buying_one_in_the_next_2–3_years",average_daily_driving_distance,...,ev_resale_value_is_uncertain,limited_service_centers,preferred_ev_price_range,minimum_acceptable_driving_range_per_charge,important_factors_when_choosing_an_ev,have_you_ever_sold_or_considered_selling_an_ev,primary_reason_for_resale___consideration,do_you_believe_evs_depreciate_faster_than_petrol_or_diesel_vehicles,does_strong_ev_sales_increase_your_trust_in_a_company,would_declining_ev_sales_reduce_your_confidence_in_an_ev_company’s_future
0,Bahrain,Urban,26-35,Average national income,Salaried employee,yes,Petrol / Diesel,none,Yes,20–50 km,...,5.0,5.0,Mid-range (9 Lakhs - 20 Lakhs),200–300 km,Driving range,Yes,Financial reasons,3,3,3
1,India,Semi-urban,46-60,Above average national income,Homemaker,yes,Petrol / Diesel,Hyundai,Maybe,Less than 20 km,...,5.0,5.0,Mid-range (9 Lakhs - 20 Lakhs),300–400 km,Price;Driving range;Charging speed,No,Upgrade to newer model,3,5,5
2,India,Semi-urban,46-60,Above average national income,Salaried employee,yes,Petrol / Diesel,none,Maybe,Less than 20 km,...,2.0,4.0,Premium ( 21 Lakhs - 39 Lakhs),<200 km,Price;Driving range;Brand;Design & comfort,NaN,not_applicable,3,4,5
3,United States,Urban,46-60,Above average national income,Salaried employee,yes,Petrol / Diesel,Kia,Yes,Less than 20 km,...,2.0,2.0,Premium ( 21 Lakhs - 39 Lakhs),300–400 km,Price;Driving range,No,Upgrade to newer model,4,4,4
4,India,Urban,46-60,Average national income,Salaried employee,yes,Petrol / Diesel,none,Yes,More than 100 km,...,5.0,5.0,Mid-range (9 Lakhs - 20 Lakhs),400 km,Price;Driving range;Charging speed;Brand;Desig...,No,Battery degradation,5,5,4


In [63]:
yes_no_map = {
"yes": 1,
"no": 0,
"maybe": 0.5
}

yes_no_cols = [
"do_you_currently_own_a_vehicle",
"if_you_do_not_own_an_ev,_are_you_considering_buying_one_in_the_next_2–3_years",
"have_you_ever_sold_or_considered_selling_an_ev"
]

for col in yes_no_cols:
    survey_df[col] = survey_df[col].str.lower().map(yes_no_map)

In [64]:
area_map = {
"urban":3,
"semi-urban":2,
"rural":1
}

survey_df["area_type"] = survey_df["area_type"].str.lower().map(area_map)

In [65]:
age_map = {
"18-25":1,
"26-35":2,
"36-45":3,
"46-60":4,
"60+":5
}

survey_df["age_group"] = survey_df["age_group"].map(age_map)

In [66]:
distance_map = {
"Less than 20 km":1,
"20–50 km":2,
"50–100 km":3,
"More than 100 km":4
}

survey_df["average_daily_driving_distance"] = \
survey_df["average_daily_driving_distance"].map(distance_map)

In [67]:
range_map = {
"<200 km":1,
"200–300 km":2,
"300–400 km":3,
"400 km":4
}

survey_df["minimum_acceptable_driving_range_per_charge"] = \
survey_df["minimum_acceptable_driving_range_per_charge"].map(range_map)

In [68]:
price_map = {
"Budget":1,
"Mid-range (9 Lakhs - 20 Lakhs)":2,
"Premium (21 Lakhs - 39 Lakhs)":3
}

survey_df["preferred_ev_price_range"] = \
survey_df["preferred_ev_price_range"].map(price_map)

In [70]:
motivation_cols = [
"lower_running_cost",
"environmental_benefits",
"government_incentives___subsidies",
"rising_fuel_prices",
"advanced_technology_&_features",
"brand_reputation",
"charging_convenience"
]

# convert columns to numeric
survey_df[motivation_cols] = survey_df[motivation_cols].apply(pd.to_numeric, errors="coerce")

# now compute the score
survey_df["ev_motivation_score"] = survey_df[motivation_cols].mean(axis=1)

In [72]:
barrier_cols = [
"evs_are_too_expensive",
"charging_infrastructure_is_insufficient",
"battery_replacement_cost_is_high",
"driving_range_is_inadequate",
"charging_time_is_too_long",
"ev_resale_value_is_uncertain",
"limited_service_centers"
]

# convert to numeric
survey_df[barrier_cols] = survey_df[barrier_cols].apply(pd.to_numeric, errors="coerce")

# calculate barrier score
survey_df["ev_barrier_score"] = survey_df[barrier_cols].mean(axis=1)

In [73]:
survey_df["factor_price"] = survey_df["important_factors_when_choosing_an_ev"].str.contains("Price").astype(int)

survey_df["factor_range"] = survey_df["important_factors_when_choosing_an_ev"].str.contains("Driving range").astype(int)

survey_df["factor_brand"] = survey_df["important_factors_when_choosing_an_ev"].str.contains("Brand").astype(int)

survey_df["factor_design"] = survey_df["important_factors_when_choosing_an_ev"].str.contains("Design").astype(int)

survey_df["factor_charging"] = survey_df["important_factors_when_choosing_an_ev"].str.contains("Charging").astype(int)

In [74]:
survey_df["ev_adoption_score"] = (
survey_df["ev_motivation_score"] -
survey_df["ev_barrier_score"]
)

In [75]:
survey_augmented = survey_df.sample(
n=500,
replace=True,
random_state=42
)

survey_augmented.to_csv(
"../data/processed/survey_augmented.csv",
index=False
)

In [76]:
survey_df.to_csv(
"../data/processed/survey_clean.csv",
index=False
)

In [77]:
survey_df.shape

(119, 40)

In [80]:
target_size = 325

survey_augmented = survey_df.sample(
    n=target_size,
    replace=True,
    random_state=42
).reset_index(drop=True)

In [81]:
survey_augmented.shape

(325, 40)

In [83]:
survey_augmented["ev_readiness_score"] = (
    survey_augmented["ev_motivation_score"] -
    survey_augmented["ev_barrier_score"]
)

In [84]:
import numpy as np

noise_cols = [
    "ev_motivation_score",
    "ev_barrier_score",
    "ev_readiness_score"
]

for col in noise_cols:
    survey_augmented[col] = survey_augmented[col] + np.random.normal(
        0,
        0.15,
        size=len(survey_augmented)
    )

In [85]:
for col in noise_cols:
    survey_augmented[col] = survey_augmented[col].clip(1,5)

In [86]:
survey_augmented.describe()

,area_type,age_group,do_you_currently_own_a_vehicle,"if_you_do_not_own_an_ev,_are_you_considering_buying_one_in_the_next_2–3_years",average_daily_driving_distance,lower_running_cost,environmental_benefits,government_incentives___subsidies,rising_fuel_prices,advanced_technology_&_features,...,would_declining_ev_sales_reduce_your_confidence_in_an_ev_company’s_future,ev_motivation_score,ev_barrier_score,factor_price,factor_range,factor_brand,factor_design,factor_charging,ev_adoption_score,ev_readiness_score
count,325.000000,313.000000,325.000000,325.000000,325.000000,179.000000,187.000000,187.000000,186.000000,186.000000,...,325.000000,187.000000,313.000000,325.000000,325.000000,325.000000,325.000000,325.000000,179.000000,179.000000
mean,2.440000,2.792332,0.910769,0.581538,2.024615,3.770950,3.631016,3.069519,3.951613,4.064516,...,3.563077,3.676860,3.624405,0.612308,0.760000,0.393846,0.547692,0.587692,-0.020218,1.089562
std,0.720254,1.131571,0.285516,0.368887,0.892014,1.169867,1.370857,1.379762,1.384401,1.325887,...,1.214568,1.021502,0.950222,0.487975,0.427742,0.489355,0.498488,0.493009,1.172642,0.272729
min,1.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-3.428571,1.000000
25%,2.000000,2.000000,1.000000,0.500000,1.000000,3.000000,3.000000,2.000000,3.000000,4.000000,...,3.000000,3.128101,2.991005,0.000000,1.000000,0.000000,0.000000,0.000000,-0.571429,1.000000
50%,3.000000,3.000000,1.000000,0.500000,2.000000,4.000000,4.000000,3.000000,5.000000,5.000000,...,4.000000,3.933620,3.776974,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,1.000000
75%,3.000000,4.000000,1.000000,1.000000,3.000000,5.000000,5.000000,4.000000,5.000000,5.000000,...,5.000000,4.381735,4.368545,1.000000,1.000000,1.000000,1.000000,1.000000,0.714286,1.000000
max,3.000000,4.000000,1.000000,1.000000,4.000000,5.000000,5.000000,5.000000,5.000000,5.000000,...,5.000000,5.000000,5.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.857143,2.724152


In [87]:
survey_augmented.to_csv(
"../data/processed/survey_clean.csv",
index=False
)